In [1]:
from swemnics.problems import SlopedBeachProblem
from swemnics import solvers as Solvers
from mpi4py import MPI
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dolfinx import fem as fe
import pickle
from tqdm import tqdm

import pandas as pd
from scipy.optimize import minimize
import seaborn as sns
from typing import Callable, Dict, List, Tuple, Any, Union

from plotting import plot_simulation_results, create_comparison_figure
from fourd_var import run_assimilation
from dca_utils import*

sns.set_palette("bright")
plt.style.use("mystyle1.mplstyle")


In [2]:
comm = MPI.COMM_WORLD
rank = comm.Get_rank()

problem_params = {
    'dt': 600,
    't': 0,
    't_final': 7*24*60*60,
    'num_steps': int(np.ceil((7*24*60*60)/600)),
    'num_windows': 4,
    'fric_law': 'mannings',    #friction law either quadratic or linear
    'sol_var': 'h'             #solution variable either h or hu
}

solver_params = {"rtol": 1e-5,
          "atol": 1e-6,
          "max_it":10,
          "relaxation_parameter":1.0,
          "ksp_type": "gmres",
          "pc_type": "ilu",
          "ksp_ErrorIfNotConverged": False
          }#,"pc_factor_mat_solver_type":"mumps"}



In [3]:
assert problem_params['num_steps'] == int(np.ceil(problem_params['t_final']/problem_params['dt']))
true_signal, prob, stations, state_coords = get_true_signal(problem_params,'sloped_beach', solver_params,4)

Location of shoreline: 13800
Fully conservative flux
Fully conservative flux
WD nonspherical

Fully conservative flux
V_coords shape: (432, 3)


In [4]:
type(prob), type(true_signal.V)

(swemnics.problems.SlopedBeachProblem, dolfinx.fem.function.FunctionSpace)

In [5]:

def build_observation_matrix(prob, V, obs_time_freq=2):
    num_cells = len(prob.mesh.geometry.dofmap)
    all_cells = np.arange(num_cells)
    obs_space_idx = np.arange(0, num_cells, obs_time_freq) # select every obs_time_freq-th cell for observation
    station_cells = all_cells[obs_space_idx]  # select cells for observation

    # Create observation matrix
    H = np.zeros((len(station_cells), V.dofmap.index_map.size_local))

    # pick subset of cells for the stations 
    station_coords = []

    # collapse the function space to get the coordinates of the dofs
    # in the cells that are selected for observation
    V_collapsed, indices_into_V = V.sub(0).collapse()   
    collapsed_dof_coords = V_collapsed.tabulate_dof_coordinates()
    indices_into_V = np.array(indices_into_V)

    for station, i in enumerate(station_cells):
        coords_for_cell = collapsed_dof_coords[V_collapsed.dofmap.cell_dofs(i)]
        dofs_in_orig_V = indices_into_V[V_collapsed.dofmap.cell_dofs(i)]
        H[station, dofs_in_orig_V] = 1/3 
        station_coord = 1/3 * (coords_for_cell.sum(axis=0))
        station_coords.append(station_coord)
        # print(f"Station {station} at {station_coord} corresponds to cell {i} with dofs {dofs_in_orig_V}")
    
    return H, np.array(station_coords), obs_space_idx


def generate_observations(true_states, H, obs_time_idx, obs_std=0.1):
    # Extract only the states at the observation indices
    true_states = np.array(true_states)  # Ensure true_states is a numpy array
    observed_states = true_states[obs_time_idx]  # shape: (n_obs, state_dim)
    
    # Apply observation operator to all observed states at once
    y_n = observed_states @ H.T  # shape: (n_obs, obs_dim)

    # Add Gaussian noise
    noise = obs_std * np.random.randn(*y_n.shape)
    y_obs = y_n + noise

    return y_obs

def setup_observation_indices(window_size, obs_frequency, total_steps):
    """Setup observation indices for windows"""
    obs_indices_per_window = np.arange(0, window_size, obs_frequency)
    obs_indices = np.arange(0, total_steps - 1, obs_frequency)
    return obs_indices_per_window, obs_indices

In [6]:

obs_std=0.4
obs_time_freq = 2
obs_space_freq = 2
total_steps = int((problem_params['t_final']/problem_params['dt']) + 1)
problem_params['num_steps'] = int(np.ceil((7*24*60*60)/600)/problem_params['num_windows']) # Size of each assimilation window
obs_per_window = problem_params['num_steps'] // obs_time_freq

H, stations, obs_spatial_indices = build_observation_matrix(prob, true_signal.V, obs_space_freq)
obs_indices_per_window, obs_time_indices = setup_observation_indices(problem_params['num_steps'], obs_time_freq, total_steps)

# Create synthetic observations
y_obs = generate_observations(true_signal.saved_states, H, obs_time_indices, obs_std)

print(f"Total Steps: {total_steps}\n"
      f"Total Assimilation Windows: {problem_params['num_windows']}\n"
      f"Steps per Window: {problem_params['num_steps']}\n"
      f"Obs Frequency: {obs_time_freq}\n"
      f"Total Obs: {obs_per_window * problem_params['num_windows']}\n"
      f"Number Stations: {stations.shape[0]}\n"
      f"Obs per Window: {obs_per_window}\n")



# Generate Background,Observation, and Predicted Error Covariance Matrices
state_dim = true_signal.saved_adjoints[0].shape[0]
obs_dim = stations.shape[0]

# Observation Covariance 
R = np.eye(obs_dim) * (obs_std**2)

inflation_factor=2.0
B = inflation_factor*np.eye(state_dim) 

# Predicted Covariance
L = H @ B @ H.T 

# Get Inverse Covariance matrices
R_inv = np.linalg.inv(R)
B_inv = np.linalg.inv(B) 
L_inv = np.linalg.inv(L)

covs = {"B_inv": B_inv, "R_inv": R_inv, "L_inv": L_inv}

hb = 5.0 / 13800 * (13800 - stations[:, 0])

print(f"State Dimension: {state_dim}\n"
      f"Observation Dimension: {obs_dim}\n"
      f"Background Covariance Matrix Shape B: {B.shape}\n"
      f"Observation Covariance Matrix Shape R: {R.shape}\n"
      f"Predicted Error Covariance Matrix shape L: {L.shape}\n"
      f"Observation Matrix Shape H: {H.shape}\n",
      f"y_obs shape: {y_obs.shape}\n",
      f"Stations shape: {stations.shape}\n",
)


Total Steps: 1009
Total Assimilation Windows: 4
Steps per Window: 252
Obs Frequency: 2
Total Obs: 504
Number Stations: 72
Obs per Window: 126



State Dimension: 1296
Observation Dimension: 72
Background Covariance Matrix Shape B: (1296, 1296)
Observation Covariance Matrix Shape R: (72, 72)
Predicted Error Covariance Matrix shape L: (72, 72)
Observation Matrix Shape H: (72, 1296)
 y_obs shape: (504, 72)
 Stations shape: (72, 3)



In [7]:
print(f"max:{H.max()}, "
      f"min: {H.min()}, "
      f"2-norm: {np.linalg.norm(H, ord=2)}, "
      f"cond #: {np.linalg.cond(H)}, "
      f"rank: {np.linalg.matrix_rank(H)}")

max:0.3333333333333333, min: 0.0, 2-norm: 0.5773502691896257, cond #: 1.0, rank: 72


In [8]:
len(true_signal.saved_adjoints)

1008

In [9]:
# bayes_analysis = run_assimilation(
#                                   problem_params,
#                                   solver_params,
#                                   stations,
#                                   y_obs,
#                                   obs_per_window,
#                                   obs_spatial_indices,
#                                   obs_time_indices,
#                                   H,
#                                   covs,
#                                   hb,
#                                 'sloped_beach',
#                                 cost_function_type='bayes'
#                              )

# true_states = np.array(true_signal.saved_states)
# Hu_true = (H @ true_states.T)  
# pred = bayes_analysis[1:,:,0] + hb
# bayes_height_misfit_rmse = np.sqrt(np.mean((Hu_true - pred.T) ** 2))
# print(f"Bayes Analysis Height RMSE: {bayes_height_misfit_rmse}")

In [10]:
dci_analysis = run_assimilation(
                                  problem_params,
                                  solver_params,
                                  stations,
                                  y_obs,
                                  obs_per_window,
                                  obs_spatial_indices,
                                  obs_time_indices,
                                  H,
                                  covs,
                                  hb,
                                'sloped_beach',
                                cost_function_type='dci'
                             )

true_states = np.array(true_signal.saved_states)
Hu_true = (H @ true_states.T)  
pred = dci_analysis[1:,:,0] + hb
dci_height_misfit_rmse = np.sqrt(np.mean((Hu_true - pred.T) ** 2))
print(f"DCI Analysis Height RMSE: {dci_height_misfit_rmse}")

Processing windows:   0%|          | 0/4 [00:00<?, ?window/s]

Location of shoreline: 13800
Fully conservative flux
Fully conservative flux
WD nonspherical

Fully conservative flux
Solver Time 1: 0
Background shape: (252, 1296)
Q_zb shape: (72, 126)
Iteration 1: Cost = 4586.880327

Optimization completed:
  Success: True
  Status: 0
  Message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  Final cost: 4.586880e+03
  Iterations: 1
  Function evaluations: 13
  Gradient norm at solution: 4.176935e-03

------------------------------------------------------------

State comparison (subsampled):
  Initial state (every 100th entry):   [0.41666667 0.41666667 0.83333333 0.         0.         0.
 0.         0.         0.         2.91666667 3.75       3.33333333
 0.        ]

  Optimized state (every 100th entry): [ 4.16666684e-01  4.16666667e-01  8.33333329e-01 -7.93663647e-10
 -9.35558530e-10 -4.08411174e-10  1.06934028e-11 -4.19993333e-10
 -8.31510821e-10  2.91666667e+00  3.75000000e+00  3.33333333e+00
 -2.31565659e-11]

Solver Time 2: 0


Processing windows:  25%|██▌       | 1/4 [04:11<12:35, 251.86s/window]

/////////////////////////////////////// Window 1 Completed ////////////////////////////////////////////////// 


Location of shoreline: 13800
Fully conservative flux
Fully conservative flux
WD nonspherical

Fully conservative flux
Solver Time 1: 151200
Background shape: (252, 1296)
Q_zb shape: (72, 126)
Iteration 1: Cost = 4458.031283

Optimization completed:
  Success: True
  Status: 0
  Message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  Final cost: 4.458031e+03
  Iterations: 1
  Function evaluations: 18
  Gradient norm at solution: 5.195656e-03

------------------------------------------------------------

State comparison (subsampled):
  Initial state (every 100th entry):   [ 8.38260365e-01  8.35663595e-01  1.21056309e+00 -4.26527027e-01
 -1.94405041e-05 -5.49166673e-01  3.76038569e-05 -4.25329291e-01
  4.19837286e-06  3.07140226e+00  3.83597007e+00  3.45244104e+00
 -6.46845562e-01]

  Optimized state (every 100th entry): [ 8.38260360e-01  8.35663589e-01  1.21056309e+00

Processing windows:  50%|█████     | 2/4 [12:40<13:25, 402.76s/window]

...

Processing windows:  75%|███████▌  | 3/4 [20:44<07:19, 439.90s/window]

/////////////////////////////////////// Window 3 Completed ////////////////////////////////////////////////// 


Location of shoreline: 13800
Fully conservative flux
Fully conservative flux
WD nonspherical

Fully conservative flux
Solver Time 1: 453600
Background shape: (252, 1296)
Q_zb shape: (72, 126)
Iteration 1: Cost = 4486.731134

Optimization completed:
  Success: True
  Status: 0
  Message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  Final cost: 4.486731e+03
  Iterations: 1
  Function evaluations: 16
  Gradient norm at solution: 1.105296e-02

------------------------------------------------------------

State comparison (subsampled):
  Initial state (every 100th entry):   [ 8.66512422e-01  8.63896387e-01  1.23807839e+00 -4.40300081e-01
 -1.67005364e-05 -5.72029553e-01  3.88033282e-05 -4.39265115e-01
  5.51046538e-06  3.08610889e+00  3.84463798e+00  3.46412790e+00
 -6.80101082e-01]

  Optimized state (every 100th entry): [ 8.66512426e-01  8.63896389e-01  1.23807839e+00

Processing windows: 100%|██████████| 4/4 [28:06<00:00, 421.50s/window]

/////////////////////////////////////// Window 4 Completed ////////////////////////////////////////////////// 




DCI Analysis Height RMSE: 0.0022231062237823423


In [ ]:
plt.plot(Hu_true[10], linestyle="solid", label='True h')
plt.plot(pred.T[10], linestyle="dashed", label='Predicted h')
plt.legend()
plt.show()

In [ ]:
bayes_rmse = np.sqrt(np.mean((true_signal.vals[:,:,0] - bayes_analysis[:,:,0]) ** 2))
bayes_rmse

In [ ]:
plot_params = {
    "lines.linewidth": 5,
    "lines.markersize": 10,
    'lines.markeredgecolor': 'black',
    "legend.fontsize": 45,
    "legend.frameon": False,
    "xtick.labelsize": 45,
    "ytick.labelsize": 45,
    "axes.labelsize": 45,
    "axes.labelpad": 10,
    "axes.titlesize": 45,
    "figure.figsize": (44, 16),
}


plot_simulation_results(true_signal, bayes_analysis, y_obs, hb, problem_params,
                       obs_indices, plot_params, station_idx=5, save=False, save_prefix="bayes_")